E-20: two-sample inference under covariate shift (H20, `thm:cs_asymp_normality`, `rem:cs_no_ratio`)

Registration: `doc/2026-10-07_experiment_registration.md` (parent repository), E-20, §1.3 A-5 (the strict two-sample density-ratio path) and §1.4–§1.9. One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-20 stage0 17_E20_covariate_shift_two_sample.ipynb`
- `... E-20 stage0.5 ...`, then `... E-20 stage1 ...` (each after the previous stage is committed).

DGP, arms, the replication function and the aggregation are in `grrexp/e20.py`; the pilot and the confirmatory stage run the same aggregation. A solver that raises stops the notebook and the run is recorded as failed (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import baselines, e12, e20, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
rec = outputs.RunRecorder(20, STAGE, cells=None if STAGE == "stage0" else e20.CELLS,
                          arms=None if STAGE == "stage0" else [e20.record_label(a) for a in e20.ARM_LABELS])
#: Stage 1 workers (results do not depend on the number of workers; Stage 0.5 checks 1 vs 12)
STAGE1_WORKERS = 6
STAGE

'stage0'

# Stage 0: population ratio, moments, predictions and the certificate (registration E-20 Stage 0, §1.9)

In [2]:
if STAGE == "stage0":
    dgp = e20.dgp_check()
    print(json.dumps(dgp, indent=1))
    if not dgp["ok"]:
        raise RuntimeError("DGP20 does not reproduce the registered E_0 r0^2")
    pop, stage0_warnings, _ = baselines.run_recording_warnings(e20.stage0)
    rec.write_stage0("e20_population", {"population": pop, "dgp_check": dgp})
    print(pop["certified"], pop["checks"])
    print(json.dumps(pop["predictions"], indent=1))
    rec.finalize(R=None, n=None, warnings=int(sum(stage0_warnings.values())), failures=0)

{
 "E_r0_sq_closed_form": 1.543779950949559,
 "E_r0_sq_quadrature": 1.5437799509495593,
 "E_r0_sq_registered": 1.54378,
 "E_r0": 1.0,
 "ok": true
}
True {'i_gradient': True, 'i_beta_diff': True, 'iii_margin': True, 'ii_iv_positive_definite': True}
{
 "500,500": {
  "pi": 0.5,
  "V_CS": 4.2223146210862685,
  "sd": 0.0649793399557603,
  "c": {
   "V_2s": 0.950004209703559,
   "V_src": 0.906272735621017,
   "V_pool": 0.9634663583900505
  }
 },
 "2000,2000": {
  "pi": 0.5,
  "V_CS": 4.2223146210862685,
  "sd": 0.03248966997788015,
  "c": {
   "V_2s": 0.950004209703559,
   "V_src": 0.906272735621017,
   "V_pool": 0.9634663583900505
  }
 },
 "8000,8000": {
  "pi": 0.5,
  "V_CS": 4.2223146210862685,
  "sd": 0.016244834988940075,
  "c": {
   "V_2s": 0.950004209703559,
   "V_src": 0.906272735621017,
   "V_pool": 0.9634663583900505
  }
 },
 "8000,1000": {
  "pi": 0.1111111111111111,
  "V_CS": 6.843148681160428,
  "sd": 0.02757444203356198,
  "c": {
   "V_2s": 0.950004209703559,
   "V_src": 0.676

# The workflow after the replications (shared by Stage 0.5 and Stage 1)

The §1.3 C check of every successful ratio fit (stops on a violation), the §1.5 metrics per configuration, arm and variance estimator, the family-1 MCSE intervals, and the family H20 (Holm 0.01).

Bootstrap call order (stream 4, family 1 only, `e12.family1_mcse`): the rows of the summary in order (configurations in registered order, then `e20.ARM_LABELS`), within a row CI length (mean), max weight (median), SE ratio; rows with fewer than two successes consume no draws. H20 uses exact binomial tests and no bootstrap.

In [3]:
def load_population():
    return json.loads((outputs.stage_dir(20, "stage0") / "stage0_e20_population.json").read_text())["population"]


def aggregate(raw, pop, seeds):
    bal = e20.ratio_balance(raw)
    summ = e20.summarise(raw, pop)
    summ = summ.assign(**e12.family1_mcse(raw, summ, seeds))
    verdict, tests = e20.family_h20(summ)
    summ["family_verdict"] = json.dumps(verdict.sentence() if verdict else None)
    summ["family_tests"] = json.dumps(tests)
    summ["family_rejected"] = json.dumps(list(verdict.rejected) if verdict else [])
    summ["ratio_balance"] = json.dumps(bal)
    return summ

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

Per configuration and number of workers: the 10 replications and the aggregation of the configuration. The aggregation of the whole pilot sample and the table generation are timed once and recorded in the manifest. Pilot estimates and tables are discarded.

In [4]:
if STAGE == "stage0.5":
    parallel.configure_worker()
    pop = load_population()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e20.CELLS):
            t0 = time.perf_counter()
            tasks = e20.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, cells=[ci])
            res = parallel.run_tasks(e20.replicate, tasks, workers)
            aggregate(e20.raw_frame(tasks, res), pop, Seeds(20, entropy=PILOT_ENTROPY))
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e20.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    aggregate(raw, pop, Seeds(20, entropy=PILOT_ENTROPY)).to_csv(buf, index=False)
    buf.seek(0)
    S = pd.read_csv(buf)
    e20.tables(S)
    whole_seconds = time.perf_counter() - t0
    counts = raw.groupby(["cell", "arm", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e20.CELLS[c] for c in counts["cell"]],
                              "arm": counts["arm"].map(e20.record_label),
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e20.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    v2s = raw["arm"].str.endswith("|V_2s")  # one record per arm and replication, as in Stage 1
    rec.finalize(R={"main": outputs.PILOT_REPS}, n={"main": e20.CELLS},
                 warnings=e20.total_warnings(raw), failures=int((v2s & (raw["status"] != "ok")).sum()),
                 extra={"whole_pilot_aggregation_and_table_seconds": whole_seconds})
    del raw

# Stage 1: confirmatory Monte Carlo

In [5]:
if STAGE == "stage1":
    parallel.configure_worker()
    tasks = e20.tasks_for(CONFIRMATORY_ENTROPY, e20.R)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e20.replicate, tasks, STAGE1_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw = e20.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e20.total_warnings(raw)
    raw.shape, n_warn

In [6]:
if STAGE == "stage1":
    summ = aggregate(raw, load_population(), Seeds(20))  # the §1.3 C check stops the notebook on a violation
    rec.write_summary(summ)
    {k: json.loads(summ[k].iloc[0]) for k in ("ratio_balance", "family_verdict", "family_rejected")}

# Tables (from summary.csv only, §4)

In [7]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    tabs, fam = e20.tables(S)
    for name, tex in tabs.items():
        rec.write_table(name, tex)
    print(fam)
    v2s = S[S["arm"].str.endswith("|V_2s")]
    rec.finalize(R={"main": e20.R}, n={"main": e20.CELLS}, warnings=int(n_warn),
                 failures=int((v2s["R"] - v2s["R_s"]).sum()))